# 02 — Analysis preparation: one table per hypothesis

**Project question: England's comeback — a miracle or the right coaching decisions?**

Each section states a hypothesis, prepares the data, saves the table(s) to `data/processed/` and prints a
short numeric summary. There are no charts here — the tables are the input for the figures.

Conventions (see `src/data.py`):
* StatsBomb coordinates are yards on a 120 x 80 pitch; every event is recorded from the acting team's
  perspective (attacking towards x = 120). For an attacking team, **y < 40 is its left flank**
  (checked against player positions: left backs average y ≈ 13, right backs y ≈ 69).
* `minute`/`second` are the elapsed clock; `display_minute = minute + 1` is the broadcast-style minute.
* "In play" means periods 1–4; period 5 is the penalty shootout.

In [1]:
import sys
from math import comb
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import data as d

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

match_id = d.find_match_id()
events = d.load_events(match_id)
lineups = d.load_lineups(match_id)
frames = d.load_frames_360(match_id)
names = d.player_display_names(lineups)

events["display_minute"] = events.minute + 1
events["player_name"] = events.player.map(names).fillna(events.player)
in_play = events[events.period < d.SHOOTOUT_PERIOD]
subs = d.get_substitutions(events)

# Reference point for "before / after 77'": the exact time of Kelly's substitution in the data.
kelly = subs[(subs.team == d.ENGLAND) & subs.player_on.str.contains("Kelly")].iloc[0]
SPLIT_PERIOD, SPLIT_CLOCK = int(kelly.period), kelly.minute + kelly.second / 60
SPLIT_KEY = SPLIT_PERIOD * 1000 + SPLIT_CLOCK
print(f"Split point: Kelly on for {kelly.player_off} at {int(kelly.minute)}:{int(kelly.second):02d} "
      f"(period {SPLIT_PERIOD})")

IN_PLAY_END_PERIOD = int(in_play.period.max())
IN_PLAY_END_CLOCK = float(d.clock(in_play[in_play.period == IN_PLAY_END_PERIOD]).max())
MIN_BEFORE = d.played_minutes(events, 1, 0.0, SPLIT_PERIOD, SPLIT_CLOCK)
MIN_AFTER = d.played_minutes(events, SPLIT_PERIOD, SPLIT_CLOCK, IN_PLAY_END_PERIOD, IN_PLAY_END_CLOCK)
print(f"Played minutes before split: {MIN_BEFORE}, after split (to end of extra time): {MIN_AFTER}")


def add_split_flag(df: pd.DataFrame) -> pd.DataFrame:
    """Add `after_77` = event at or after Kelly's substitution."""
    return df.assign(after_77=d.time_key(df) >= SPLIT_KEY)


def pitch_third(x: pd.Series) -> pd.Series:
    """Third of the pitch from the acting team's perspective."""
    return pd.cut(x, [-0.1, 40, 80, 120.1], labels=["defensive", "middle", "attacking"]).astype(str)

Split point: Kelly on for Lauren Hemp at 77:33 (period 2)
Played minutes before split: 79.17, after split (to end of extra time): 52.68


---
## H1 — "Sweden's press broke England's build-up in the first half"

**Data needed:** England ball losses in the first half and Sweden's tackles, interceptions and pressures,
with coordinates; the number of Sweden players close to the England player at each loss (360 data).

**Window.** "0–45'" is taken as the whole of period 1, *including* first-half stoppage time
(the half ends at 46:37 on the StatsBomb clock).

**Definition of a ball loss (England event, period 1):**

| Event | Condition | Counted? |
|---|---|---|
| `Pass` | `pass_outcome` in `Incomplete`, `Out`, `Pass Offside` | yes |
| `Pass` | `pass_outcome == "Unknown"` | **no** — outcome not determined by StatsBomb |
| `Miscontrol` | always | yes |
| `Dispossessed` | always | yes |
| `Dribble` | `dribble_outcome == "Incomplete"` | yes |
| `Ball Receipt*` incomplete | — | no — the receiving side of an incomplete pass (double count) |
| `Error` | — | no — a tag that accompanies another event (double count) |
| `Shot`, `Clearance`, `Duel` | — | no — a shot or a clearance ends a possession by choice; a lost `Tackle` duel is recorded for the *tackling* team and appears as `Dispossessed` for the loser |

All passes are kept, including set pieces (`pass_type` is in the table for filtering).
Not every "loss" changes possession (e.g. an incomplete pass can be recovered by England straight away),
so the table also has `turnover_confirmed`, which requires all three of:
1. the loss happens inside an **England possession** (StatsBomb `possession_team`),
2. it is England's **last on-ball action** in that possession (no later England `Pass`, `Carry`, `Dribble`,
   `Shot`, `Miscontrol` or `Dispossessed` in the same possession) — otherwise England won the ball straight back,
3. the **next possession** in StatsBomb's `possession` sequence belongs to Sweden.

**Sweden defensive actions:** `Pressure` events, `Duel` with `duel_type == "Tackle"` and `Interception`
events in period 1. Coordinates are kept in Sweden's frame (`x`, `y`) and also mirrored into England's
frame (`x_eng_frame = 120 - x`, `y_eng_frame = 80 - y`) so both tables can be drawn on the same pitch.

**360 pressure count:** Sweden players within `RADIUS_YD = 5` yards of the event location in the freeze frame
(in a frame, `teammate == False` means an opponent of the actor, i.e. a Sweden player).
Frames only contain players inside the camera's visible area, so the count is a lower bound.

In [2]:
LOSS_PASS_OUTCOMES = ["Incomplete", "Out", "Pass Offside"]
RADIUS_YD = 5.0


def england_losses(ev: pd.DataFrame) -> pd.DataFrame:
    """England ball losses in `ev` according to the definition above."""
    eng = ev[ev.team == d.ENGLAND]
    is_pass_loss = (eng.type == "Pass") & eng.pass_outcome.isin(LOSS_PASS_OUTCOMES)
    is_loss = (
        is_pass_loss
        | eng.type.isin(["Miscontrol", "Dispossessed"])
        | ((eng.type == "Dribble") & (eng.dribble_outcome == "Incomplete"))
    )
    out = eng[is_loss].copy()
    out["loss_type"] = np.select(
        [out.type.eq("Pass") & out.pass_outcome.eq("Incomplete"),
         out.type.eq("Pass") & out.pass_outcome.eq("Out"),
         out.type.eq("Pass") & out.pass_outcome.eq("Pass Offside"),
         out.type.eq("Dribble")],
        ["Pass incomplete", "Pass out of play", "Pass offside", "Dribble failed"],
        default=out.type,
    )
    return out


# Possession owner of the *next* possession, to confirm real turnovers.
poss_team = events.groupby("possession").possession_team.first()
next_poss_team = poss_team.shift(-1)

first_half = events[events.period == 1]
h1_losses = england_losses(first_half)
ON_BALL_TYPES = ["Pass", "Carry", "Dribble", "Shot", "Miscontrol", "Dispossessed"]
eng_on_ball = events[(events.team == d.ENGLAND) & events.type.isin(ON_BALL_TYPES)]
last_eng_on_ball = eng_on_ball.groupby("possession")["index"].max()

h1_losses["next_possession_team"] = h1_losses.possession.map(next_poss_team)
h1_losses["last_action_of_possession"] = h1_losses["index"] >= h1_losses.possession.map(last_eng_on_ball)
h1_losses["turnover_confirmed"] = (
    h1_losses.possession_team.eq(d.ENGLAND)
    & h1_losses.last_action_of_possession
    & h1_losses.next_possession_team.eq(d.SWEDEN)
)
h1_losses = d.split_xy(h1_losses, "location", "loss")
h1_losses = d.split_xy(h1_losses, "pass_end_location", "pass_end")
h1_losses["third"] = pitch_third(h1_losses.loss_x)
h1_losses["own_half"] = h1_losses.loss_x < 60
h1_losses["under_pressure"] = h1_losses.under_pressure.eq(True)

In [3]:
# 360: Sweden players near the England player at the moment of the loss.
f = frames[frames.id.isin(h1_losses.id)].copy()
f = d.split_xy(f, "location", "ff")
f = f.merge(h1_losses[["id", "loss_x", "loss_y"]], on="id")
f["dist"] = np.hypot(f.ff_x - f.loss_x, f.ff_y - f.loss_y)
opp = f[~f.teammate]
near = opp.groupby("id").agg(
    sweden_within_5yd=("dist", lambda s: int((s <= RADIUS_YD).sum())),
    nearest_sweden_yd=("dist", "min"),
    sweden_visible=("dist", "size"),
)

h1_losses["has_360"] = h1_losses.id.isin(frames.id)
h1_losses = h1_losses.merge(near, left_on="id", right_index=True, how="left")
# A frame with no visible Sweden player means zero nearby (not missing).
h1_losses.loc[h1_losses.has_360, "sweden_within_5yd"] = (
    h1_losses.loc[h1_losses.has_360, "sweden_within_5yd"].fillna(0)
)

h1_losses = h1_losses[[
    "id", "index", "period", "minute", "second", "display_minute", "player", "player_name", "position",
    "type", "loss_type", "pass_type", "pass_height", "play_pattern", "under_pressure",
    "loss_x", "loss_y", "pass_end_x", "pass_end_y", "third", "own_half",
    "possession", "possession_team", "last_action_of_possession", "next_possession_team", "turnover_confirmed",
    "has_360", "sweden_within_5yd", "nearest_sweden_yd", "sweden_visible",
]].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h1_losses, "h1_england_losses_first_half")))
h1_losses.head()

saved: data/processed/h1_england_losses_first_half.parquet


,id,index,period,minute,second,display_minute,player,player_name,position,type,loss_type,pass_type,pass_height,play_pattern,under_pressure,loss_x,loss_y,pass_end_x,pass_end_y,third,own_half,possession,possession_team,last_action_of_possession,next_possession_team,turnover_confirmed,has_360,sweden_within_5yd,nearest_sweden_yd,sweden_visible
0,24e0de85-686f-4e0e-9199-500405c6fd3b,9,1,0,5,1,Lucy Bronze,Lucy Bronze,Right Back,Pass,Pass incomplete,Recovery,High Pass,From Kick Off,False,32.6,76.3,43.3,74.4,defensive,True,2,Sweden Women's,True,England Women's,False,True,0.0,6.198060,10.0
1,86e77a29-0e9e-4a11-8252-592738bd13af,31,1,0,19,1,Jessica Carter,Jessica Carter,Left Center Back,Pass,Pass out of play,NaN,Ground Pass,Regular Play,False,17.9,45.5,17.4,0.4,defensive,True,3,England Women's,True,Sweden Women's,True,True,1.0,1.453135,6.0
2,9a6ba3f3-038d-4283-a33e-d0099485cb81,52,1,1,11,2,Lauren James,Lauren James,Right Wing,Pass,Pass incomplete,NaN,Low Pass,From Goal Kick,True,82.8,67.8,85.1,63.2,attacking,False,5,England Women's,True,Sweden Women's,True,True,2.0,0.887520,8.0
3,c013a576-d108-4c6b-a986-97d1fabba34b,72,1,1,39,2,Keira Walsh,Keira Walsh,Right Defensive Midfield,Pass,Pass incomplete,NaN,Ground Pass,From Goal Kick,False,32.0,29.9,35.2,29.9,defensive,True,7,England Women's,True,Sweden Women's,True,True,1.0,2.811218,6.0
4,94258e0f-c482-45eb-aae6-018b0ca11db8,88,1,2,48,3,Lucy Bronze,Lucy Bronze,Right Back,Pass,Pass incomplete,NaN,Low Pass,From Kick Off,True,84.5,75.1,85.3,74.2,attacking,False,9,England Women's,False,Sweden Women's,False,True,2.0,0.631202,9.0


In [4]:
SWE_DEF_TYPES = ["Pressure", "Interception"]
swe = first_half[first_half.team == d.SWEDEN]
is_tackle = (swe.type == "Duel") & (swe.duel_type == "Tackle")
h1_sweden_def = swe[swe.type.isin(SWE_DEF_TYPES) | is_tackle].copy()
h1_sweden_def["action"] = np.where(h1_sweden_def.type.eq("Duel"), "Tackle", h1_sweden_def.type)
h1_sweden_def["outcome"] = h1_sweden_def.duel_outcome.fillna(h1_sweden_def.interception_outcome)
h1_sweden_def["counterpress"] = h1_sweden_def.counterpress.eq(True)
h1_sweden_def = d.split_xy(h1_sweden_def, "location", "def")
h1_sweden_def["x_eng_frame"] = 120 - h1_sweden_def.def_x
h1_sweden_def["y_eng_frame"] = 80 - h1_sweden_def.def_y
h1_sweden_def["third_sweden_frame"] = pitch_third(h1_sweden_def.def_x)
h1_sweden_def = h1_sweden_def[[
    "id", "index", "period", "minute", "second", "display_minute", "player", "player_name", "position",
    "action", "outcome", "counterpress", "duration",
    "def_x", "def_y", "x_eng_frame", "y_eng_frame", "third_sweden_frame",
]].rename(columns={"def_x": "x", "def_y": "y"}).reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h1_sweden_def, "h1_sweden_defensive_actions_first_half")))
h1_sweden_def.head()

saved: data/processed/h1_sweden_defensive_actions_first_half.parquet


,id,index,period,minute,second,display_minute,player,player_name,position,action,outcome,counterpress,duration,x,y,x_eng_frame,y_eng_frame,third_sweden_frame
0,5f7e2433-2fbf-4b6c-aa28-9e336007dcb9,16,1,0,11,1,Kosovare Asllani,Kosovare Asllani,Center Attacking Midfield,Pressure,NaN,True,0.310880,90.4,2.8,29.6,77.2,attacking
1,74324960-0a38-4e59-8f2b-82d3498ed9e8,18,1,0,12,1,Fridolina Rolfö,Fridolina Rolfö,Left Wing,Pressure,NaN,True,0.184496,88.6,2.8,31.4,77.2,attacking
2,2837d652-4048-4fc4-86fa-9d8e501bf061,30,1,0,19,1,Kosovare Asllani,Kosovare Asllani,Center Attacking Midfield,Pressure,NaN,False,0.298993,100.0,31.8,20.0,48.2,attacking
3,92689959-c4e6-4f9b-aa2f-c0b6dfc27433,48,1,1,10,2,Jonna Ann-Charlotte Andersson,Jonna Andersson,Left Back,Pressure,NaN,False,0.687671,34.8,11.6,85.2,68.4,defensive
4,e484384b-6742-4e24-97f3-b8baf3ab25f3,51,1,1,11,2,Ingrid Filippa Angeldal,Ingrid Filippa Angeldal,Right Defensive Midfield,Pressure,NaN,False,0.241186,41.8,15.8,78.2,64.2,middle


### H1 — numeric summary

In [5]:
second_half_losses = england_losses(events[events.period == 2])
print(f"England ball losses: 1st half {len(h1_losses)}, 2nd half {len(second_half_losses)} (context)")
print(f"Confirmed turnovers to Sweden (1st half): {h1_losses.turnover_confirmed.sum()} "
      f"({h1_losses.turnover_confirmed.mean():.0%})")
print(f"Losses in England's own half: {h1_losses.own_half.sum()}; under pressure: {h1_losses.under_pressure.sum()}")
print(f"Losses with a 360 frame: {h1_losses.has_360.sum()} / {len(h1_losses)}")
print(f"Mean Sweden players within {RADIUS_YD:.0f} yd (360 losses): {h1_losses.sweden_within_5yd.mean():.2f}")

pd.crosstab(h1_losses.loss_type, h1_losses.third, margins=True)

England ball losses: 1st half 81, 2nd half 82 (context)
Confirmed turnovers to Sweden (1st half): 36 (44%)
Losses in England's own half: 32; under pressure: 19
Losses with a 360 frame: 68 / 81
Mean Sweden players within 5 yd (360 losses): 0.91


third,attacking,defensive,middle,All
loss_type,,,,
Dispossessed,3,2,1,6
Dribble failed,2,0,1,3
Miscontrol,4,1,3,8
Pass incomplete,18,16,21,55
Pass out of play,3,3,3,9
All,30,22,29,81


In [6]:
pd.crosstab(h1_sweden_def.action, h1_sweden_def.third_sweden_frame, margins=True)

third_sweden_frame,attacking,defensive,middle,All
action,,,,
Interception,0,2,0,2
Pressure,14,38,27,79
Tackle,2,5,2,9
All,16,45,29,90


In [7]:
h1_losses.groupby("player_name").size().sort_values(ascending=False).rename("losses").head(8)

player_name
Lucy Bronze        13
Lauren Hemp        12
Alex Greenwood     10
Leah Williamson    10
Ella Toone          7
Georgia Stanway     6
Keira Walsh         6
Lauren James        5
Name: losses, dtype: int64

### H1 — confirmed turnovers only

Subset of the losses above with `turnover_confirmed` (see the definition at the top of H1).
`defensive_third_loss` flags turnovers in England's defensive third (x < 40 in England's attacking frame) —
the zone where a broken build-up is most dangerous.

**Did the turnover hurt England?** For each turnover the first Sweden shot that satisfies either condition is linked:
* it is in the **same possession** Sweden started with the turnover (the next StatsBomb `possession` number), or
* it comes **within `SHOT_WINDOW_S = 20` seconds** of the turnover in the same period.

Time is measured from the period-relative `timestamp` (millisecond precision).
`led_to_sweden_shot`, `shot_xg`, `seconds_to_shot`, `n_linked_shots` (all Sweden shots linked by the same rule)
and `led_to_goal` (the linked shot was a Sweden goal) are added. `loss_type_description` explains each loss type.

In [8]:
SHOT_WINDOW_S = 20.0
LOSS_TYPE_DESCRIPTIONS = {
    "Pass incomplete": "Incomplete pass: the pass did not reach a teammate and the ball stayed in play",
    "Pass out of play": "Pass out of play: the pass went over the touchline or goal line",
    "Pass offside": "Offside pass: the intended receiver was caught offside",
    "Miscontrol": "Miscontrol: the player lost the ball through a poor touch",
    "Dispossessed": "Dispossessed: the player was tackled and lost the ball without attempting a dribble",
    "Dribble failed": "Failed dribble: the player tried to beat an opponent and lost the ball",
}


def timestamp_seconds(ts: pd.Series) -> pd.Series:
    """Period-relative StatsBomb timestamp 'HH:MM:SS.mmm' -> seconds."""
    return pd.to_timedelta(ts).dt.total_seconds()


ev_t = events.assign(t=timestamp_seconds(events.timestamp))
swe_shots_all = ev_t[(ev_t.type == "Shot") & (ev_t.team == d.SWEDEN) & (ev_t.period < d.SHOOTOUT_PERIOD)]

h1_turnovers = h1_losses[h1_losses.turnover_confirmed].copy()
h1_turnovers["defensive_third_loss"] = h1_turnovers.loss_x < 40
h1_turnovers["loss_type_description"] = h1_turnovers.loss_type.map(LOSS_TYPE_DESCRIPTIONS)
h1_turnovers = h1_turnovers.merge(ev_t[["id", "t"]], on="id", how="left")
poss_ids = np.sort(events.possession.unique())
next_poss_id = dict(zip(poss_ids[:-1], poss_ids[1:]))
h1_turnovers["sweden_possession"] = h1_turnovers.possession.map(next_poss_id)  # Sweden's (confirmed)

linked = []
for r in h1_turnovers.itertuples():
    s = swe_shots_all[swe_shots_all.period == r.period]
    dt = s.t - r.t
    hit = s[(s.possession == r.sweden_possession) | ((dt >= 0) & (dt <= SHOT_WINDOW_S))]
    hit = hit[hit["index"] > r.index].sort_values("index")
    first = hit.iloc[0] if len(hit) else None
    linked.append({
        "led_to_sweden_shot": first is not None,
        "n_linked_shots": len(hit),
        "shot_id": None if first is None else first.id,
        "shot_player": None if first is None else first.player,
        "shot_display_minute": np.nan if first is None else first.minute + 1,
        "shot_xg": np.nan if first is None else first.shot_statsbomb_xg,
        "seconds_to_shot": np.nan if first is None else round(first.t - r.t, 1),
        "shot_outcome": None if first is None else first.shot_outcome,
        "led_to_goal": first is not None and first.shot_outcome == "Goal",
    })
h1_turnovers = pd.concat([h1_turnovers, pd.DataFrame(linked)], axis=1)
h1_turnovers = h1_turnovers.drop(columns="t").reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h1_turnovers, "h1_england_turnovers_first_half")))

print(f"Confirmed turnovers: {len(h1_turnovers)} of {len(h1_losses)} losses")
print(f"  in England's defensive third: {h1_turnovers.defensive_third_loss.sum()}")
print(f"  under pressure: {h1_turnovers.under_pressure.sum()}")
print(f"  with 360 frame: {h1_turnovers.has_360.sum()}, mean Sweden players within "
      f"{RADIUS_YD:.0f} yd: {h1_turnovers.sweden_within_5yd.mean():.2f}")
print(f"  defensive-third turnovers, mean Sweden players within {RADIUS_YD:.0f} yd: "
      f"{h1_turnovers.loc[h1_turnovers.defensive_third_loss, 'sweden_within_5yd'].mean():.2f}")
print(f"  passes out of play: {(h1_turnovers.loss_type == 'Pass out of play').sum()}")

shot_led = h1_turnovers[h1_turnovers.led_to_sweden_shot]
print(f"\nTurnovers followed by a Sweden shot (same possession or <= {SHOT_WINDOW_S:.0f} s): {len(shot_led)}")
print(f"  xG of those shots: {shot_led.shot_xg.sum():.3f}; median seconds to shot: {shot_led.seconds_to_shot.median():.1f}")
print(f"  led to a goal: {h1_turnovers.led_to_goal.sum()} turnover(s), "
      f"{h1_turnovers.loc[h1_turnovers.led_to_goal, 'shot_id'].nunique()} distinct goal(s)")
swe_goals_h1 = swe_shots_all[(swe_shots_all.period == 1) & (swe_shots_all.shot_outcome == "Goal")]
for g in swe_goals_h1.itertuples():
    src = h1_turnovers[h1_turnovers.shot_id == g.id]
    via = (f"after {src.iloc[0].player_name}'s {src.iloc[0].loss_type.lower()} at {src.iloc[0].display_minute}' "
           f"({src.iloc[0].seconds_to_shot:.1f} s)") if len(src) else "not linked to an England turnover"
    print(f"  Sweden goal {g.minute + 1}' ({g.player}): {via}")
linked_ids = set(h1_turnovers.shot_id.dropna())
print(f"  Sweden first-half shots linked to a turnover: {len(linked_ids)} of "
      f"{int((swe_shots_all.period == 1).sum())}")

pd.crosstab(h1_turnovers.loss_type, h1_turnovers.third, margins=True)

saved: data/processed/h1_england_turnovers_first_half.parquet
Confirmed turnovers: 36 of 81 losses
  in England's defensive third: 9
  under pressure: 6
  with 360 frame: 29, mean Sweden players within 5 yd: 0.90
  defensive-third turnovers, mean Sweden players within 5 yd: 0.57
  passes out of play: 8

Turnovers followed by a Sweden shot (same possession or <= 20 s): 4
  xG of those shots: 0.766; median seconds to shot: 13.0
  led to a goal: 2 turnover(s), 2 distinct goal(s)
  Sweden goal 2' (Kosovare Asllani): after Keira Walsh's pass incomplete at 2' (4.8 s)
  Sweden goal 25' (Emma Stina Blackstenius): after Lauren Hemp's pass out of play at 25' (22.8 s)
  Sweden first-half shots linked to a turnover: 4 of 4


third,attacking,defensive,middle,All
loss_type,,,,
Dispossessed,0,0,1,1
Dribble failed,1,0,0,1
Miscontrol,2,1,3,6
Pass incomplete,7,6,7,20
Pass out of play,3,2,3,8
All,13,9,14,36


In [9]:
h1_turnovers.groupby("loss_type").agg(
    turnovers=("id", "size"), led_to_shot=("led_to_sweden_shot", "sum"),
    shot_xg=("shot_xg", "sum"), led_to_goal=("led_to_goal", "sum"),
).round(3)

,turnovers,led_to_shot,shot_xg,led_to_goal
loss_type,,,,
Dispossessed,1,0,0.000,0
Dribble failed,1,0,0.000,0
Miscontrol,6,1,0.136,0
Pass incomplete,20,2,0.345,1
Pass out of play,8,1,0.286,1


In [10]:
h1_turnovers[h1_turnovers.led_to_sweden_shot][[
    "display_minute", "player_name", "loss_type", "third", "loss_x", "loss_y",
    "shot_player", "shot_display_minute", "seconds_to_shot", "shot_xg", "shot_outcome", "led_to_goal",
]]

,display_minute,player_name,loss_type,third,loss_x,loss_y,shot_player,shot_display_minute,seconds_to_shot,shot_xg,shot_outcome,led_to_goal
2,2,Keira Walsh,Pass incomplete,defensive,32.0,29.9,Kosovare Asllani,2.0,4.8,0.262062,Goal,True
4,5,Hannah Hampton,Miscontrol,defensive,18.5,37.7,Emma Stina Blackstenius,5.0,6.5,0.135633,Blocked,False
17,25,Lauren Hemp,Pass out of play,middle,64.2,13.3,Emma Stina Blackstenius,25.0,22.8,0.286011,Goal,True
35,46,Ella Toone,Pass incomplete,attacking,111.3,24.3,Fridolina Rolfö,46.0,19.5,0.082706,Saved,False


### H1 — pressure on the turnovers

`under_pressure` is StatsBomb's flag that the on-ball player was being pressed when the event happened.
It is set on the *loss event itself*, so a player can be pressed while receiving or carrying the ball and
still release an unpressured pass — the goal sequences below show the flag event by event.

In [11]:
print(f"Confirmed turnovers under pressure: {h1_turnovers.under_pressure.sum()} of {len(h1_turnovers)} "
      f"({h1_turnovers.under_pressure.mean():.0%})")
h1_turnovers[h1_turnovers.led_to_sweden_shot][[
    "display_minute", "player_name", "loss_type", "third", "under_pressure",
    "shot_player", "shot_xg", "shot_outcome", "led_to_goal",
]]

Confirmed turnovers under pressure: 6 of 36 (17%)


,display_minute,player_name,loss_type,third,under_pressure,shot_player,shot_xg,shot_outcome,led_to_goal
2,2,Keira Walsh,Pass incomplete,defensive,False,Kosovare Asllani,0.262062,Goal,True
4,5,Hannah Hampton,Miscontrol,defensive,False,Emma Stina Blackstenius,0.135633,Blocked,False
17,25,Lauren Hemp,Pass out of play,middle,False,Emma Stina Blackstenius,0.286011,Goal,True
35,46,Ella Toone,Pass incomplete,attacking,True,Fridolina Rolfö,0.082706,Saved,False


### H1 — goal sequences

Every event between two points, ordered by event index. `outcome` merges StatsBomb's outcome columns
(`pass_outcome`, `ball_receipt_outcome`, `dribble_outcome`, `duel_outcome`, `interception_outcome`,
`shot_outcome`, `goalkeeper_outcome`); an empty outcome on a `Pass` means it was completed.
`location` is in the acting team's frame; `x_eng_frame`, `y_eng_frame` put every event in England's attacking
frame so the two teams' events can be compared directly (Sweden events mirrored: `120 - x`, `80 - y`).
`t` is the period-relative timestamp in seconds.

In [12]:
OUTCOME_COLS = ["pass_outcome", "ball_receipt_outcome", "dribble_outcome", "duel_outcome",
                "interception_outcome", "shot_outcome", "goalkeeper_outcome"]


def event_sequence(first_index: int, last_index: int) -> pd.DataFrame:
    """All events with first_index <= index <= last_index, with a merged outcome and England-frame coordinates."""
    seq = ev_t[(ev_t["index"] >= first_index) & (ev_t["index"] <= last_index)].sort_values("index").copy()
    seq["outcome"] = seq[OUTCOME_COLS].bfill(axis=1).iloc[:, 0]
    seq["under_pressure"] = seq.under_pressure.eq(True)
    seq = d.split_xy(seq, "location", "loc")
    sweden = seq.team.eq(d.SWEDEN)
    seq["x_eng_frame"] = np.where(sweden, 120 - seq.loc_x, seq.loc_x)
    seq["y_eng_frame"] = np.where(sweden, 80 - seq.loc_y, seq.loc_y)
    return seq[["index", "possession", "possession_team", "timestamp", "t", "team", "player_name", "type",
                "pass_type", "pass_recipient", "outcome", "under_pressure", "location",
                "x_eng_frame", "y_eng_frame", "related_events", "id"]].reset_index(drop=True)


SEQ_COLS = ["index", "timestamp", "team", "player_name", "type", "outcome", "under_pressure",
            "location", "x_eng_frame", "y_eng_frame"]
goal_turnovers = h1_turnovers[h1_turnovers.led_to_goal].set_index("shot_display_minute")

#### 2' goal — from the start of England's possession to Asllani's shot

The check on Carter uses three independent pieces of evidence: an Asllani `Pressure` event between Carter's
receipt and Walsh's receipt, whether that pressure is linked to Carter's events through `related_events`,
and the distance between Asllani's pressure location and Carter's position (same frame).

In [13]:
g2 = goal_turnovers.loc[2]
g2_shot = ev_t[ev_t.id == g2.shot_id].iloc[0]
poss_start = int(ev_t.loc[ev_t.possession == g2.possession, "index"].min())
h1_seq_goal_2 = event_sequence(poss_start, int(g2_shot["index"]))
print("saved:", d.relative_path(d.save_processed(h1_seq_goal_2, "h1_goal_sequence_02min")))
print(f"England possession {g2.possession} starts at {h1_seq_goal_2.timestamp.iloc[0]}; "
      f"shot at {g2_shot.timestamp}; {h1_seq_goal_2.t.iloc[-1] - h1_seq_goal_2.t.iloc[0]:.1f} s in total")
h1_seq_goal_2[SEQ_COLS]

saved: data/processed/h1_goal_sequence_02min.parquet
England possession 7 starts at 00:01:32.949; shot at 00:01:44.770; 11.8 s in total


,index,timestamp,team,player_name,type,outcome,under_pressure,location,x_eng_frame,y_eng_frame
0,65,00:01:32.949,England Women's,Hannah Hampton,Pass,NaN,False,"[7.0, 38.2]",7.0,38.2
1,66,00:01:35.200,England Women's,Jessica Carter,Ball Receipt*,NaN,False,"[18.3, 36.6]",18.3,36.6
2,67,00:01:37.773,Sweden Women's,Kosovare Asllani,Pressure,NaN,False,"[95.2, 43.6]",24.8,36.4
3,68,00:01:38.053,England Women's,Jessica Carter,Pass,NaN,True,"[20.4, 33.9]",20.4,33.9
4,69,00:01:38.237,England Women's,Keira Walsh,Ball Receipt*,NaN,False,"[32.2, 30.4]",32.2,30.4
5,70,00:01:38.237,England Women's,Keira Walsh,Carry,NaN,True,"[32.2, 30.4]",32.2,30.4
6,71,00:01:38.237,Sweden Women's,Kosovare Asllani,Block,NaN,False,"[97.8, 46.0]",22.2,34.0
7,72,00:01:39.979,England Women's,Keira Walsh,Pass,Incomplete,False,"[32.0, 29.9]",32.0,29.9
8,73,00:01:40.114,England Women's,Georgia Stanway,Ball Receipt*,Incomplete,False,"[46.6, 22.9]",46.6,22.9
9,74,00:01:40.114,Sweden Women's,Ingrid Filippa Angeldal,Block,NaN,False,"[84.9, 50.2]",35.1,29.8


In [14]:
seq = h1_seq_goal_2
carter = seq[seq.player_name.str.contains("Carter", na=False) & seq.team.eq(d.ENGLAND)]
walsh_receipt = seq[seq.player_name.str.contains("Walsh", na=False) & seq.type.eq("Ball Receipt*")].iloc[0]
carter_receipt = carter[carter.type == "Ball Receipt*"].iloc[0]
carter_pass = carter[(carter.type == "Pass") & (carter.pass_recipient.str.contains("Walsh", na=False))].iloc[0]

asllani_press = seq[seq.player_name.str.contains("Asllani", na=False) & seq.type.eq("Pressure")
                    & seq["index"].between(carter_receipt["index"], walsh_receipt["index"])]
print(f"Carter receives at {carter_receipt.timestamp}, passes to Walsh at {carter_pass.timestamp} "
      f"(under_pressure={carter_pass.under_pressure}); Walsh receives at {walsh_receipt.timestamp}")
print(f"Asllani Pressure events between Carter's receipt and Walsh's receipt: {len(asllani_press)}")
for p in asllani_press.itertuples():
    linked = set(p.related_events or []) & set(carter.id)
    dist = np.hypot(p.x_eng_frame - carter_pass.x_eng_frame, p.y_eng_frame - carter_pass.y_eng_frame)
    print(f"  pressure at {p.timestamp}, {carter_pass.t - p.t:.2f} s before Carter's pass; "
          f"linked to Carter's events via related_events: {bool(linked)} "
          f"({', '.join(seq.loc[seq.id.isin(linked), 'type'])}); "
          f"distance to Carter at the pass: {dist:.1f} yd")

carter_pressed = bool(len(asllani_press)) and bool(carter_pass.under_pressure)
print(f"\nCarter pressed by Asllani before the ball reached Walsh: {'YES' if carter_pressed else 'NO / not shown in data'}")

walsh = seq[seq.player_name.str.contains("Walsh", na=False) & seq.team.eq(d.ENGLAND)]
print("Walsh's events and under_pressure flags:")
print(walsh[["timestamp", "type", "outcome", "under_pressure"]].to_string(index=False))

Carter receives at 00:01:35.200, passes to Walsh at 00:01:38.053 (under_pressure=True); Walsh receives at 00:01:38.237
Asllani Pressure events between Carter's receipt and Walsh's receipt: 1
  pressure at 00:01:37.773, 0.28 s before Carter's pass; linked to Carter's events via related_events: True (Pass); distance to Carter at the pass: 5.1 yd

Carter pressed by Asllani before the ball reached Walsh: YES
Walsh's events and under_pressure flags:
   timestamp          type    outcome  under_pressure
00:01:38.237 Ball Receipt*        NaN           False
00:01:38.237         Carry        NaN            True
00:01:39.979          Pass Incomplete           False
00:01:40.163         Error        NaN           False


#### 25' goal — from Hemp's pass out of play to Blackstenius's shot

The throw-in is the first Sweden `Pass` with `pass_type == "Throw-in"` after Hemp's pass. Sweden passes are
counted strictly between the throw-in and the shot (the throw-in itself is reported separately).

In [15]:
g25 = goal_turnovers.loc[25]
g25_shot = ev_t[ev_t.id == g25.shot_id].iloc[0]
h1_seq_goal_25 = event_sequence(int(g25["index"]), int(g25_shot["index"]))
print("saved:", d.relative_path(d.save_processed(h1_seq_goal_25, "h1_goal_sequence_25min")))
h1_seq_goal_25[SEQ_COLS]

saved: data/processed/h1_goal_sequence_25min.parquet


,index,timestamp,team,player_name,type,outcome,under_pressure,location,x_eng_frame,y_eng_frame
0,838,00:24:16.382,England Women's,Lauren Hemp,Pass,Out,False,"[64.2, 13.3]",64.2,13.3
1,839,00:24:17.824,England Women's,Georgia Stanway,Ball Receipt*,Incomplete,False,"[82.3, 8.2]",82.3,8.2
2,840,00:24:25.832,Sweden Women's,Hanna Lundkvist,Pass,NaN,False,"[27.7, 80.0]",92.3,0.0
3,841,00:24:26.199,England Women's,Georgia Stanway,Pressure,NaN,False,"[87.4, 9.7]",87.4,9.7
4,842,00:24:26.581,Sweden Women's,Ingrid Filippa Angeldal,Ball Receipt*,NaN,False,"[31.3, 71.7]",88.7,8.3
5,843,00:24:26.581,Sweden Women's,Ingrid Filippa Angeldal,Pass,NaN,False,"[31.7, 70.9]",88.3,9.1
6,844,00:24:27.151,Sweden Women's,Hanna Lundkvist,Ball Receipt*,NaN,False,"[30.7, 76.4]",89.3,3.6
7,845,00:24:27.151,Sweden Women's,Hanna Lundkvist,Carry,NaN,False,"[30.7, 76.4]",89.3,3.6
8,846,00:24:27.328,Sweden Women's,Hanna Lundkvist,Miscontrol,NaN,False,"[31.9, 77.0]",88.1,3.0
9,847,00:24:28.392,England Women's,Lauren Hemp,Ball Recovery,NaN,False,"[81.6, 8.2]",81.6,8.2


In [16]:
seq = h1_seq_goal_25
throw_in = seq[(seq.team == d.SWEDEN) & (seq.type == "Pass") & (seq.pass_type == "Throw-in")].iloc[0]
shot_row = seq.iloc[-1]
between = seq[(seq["index"] > throw_in["index"]) & (seq["index"] < shot_row["index"])]
swe_passes = between[(between.team == d.SWEDEN) & (between.type == "Pass")]
eng_on_ball_between = between[(between.team == d.ENGLAND) & between.type.isin(ON_BALL_TYPES + ["Ball Recovery"])]

print(f"Hemp's pass out of play: {seq.timestamp.iloc[0]}")
print(f"Throw-in ({throw_in.player_name}): {throw_in.timestamp}; shot ({shot_row.player_name}): {shot_row.timestamp}")
print(f"Time from Hemp's pass to the shot: {shot_row.t - seq.t.iloc[0]:.1f} s")
print(f"Time from throw-in to shot: {shot_row.t - throw_in.t:.1f} s")
print(f"Sweden passes between throw-in and shot: {len(swe_passes)} "
      f"(completed: {int(swe_passes.outcome.isna().sum())}); including the throw-in: {len(swe_passes) + 1}")
print("England on-ball events between throw-in and shot (ball won back inside Sweden's possession):")
print(eng_on_ball_between[["timestamp", "player_name", "type", "outcome", "under_pressure"]].to_string(index=False))

Hemp's pass out of play: 00:24:16.382
Throw-in (Hanna Lundkvist): 00:24:25.832; shot (Stina Blackstenius): 00:24:39.133
Time from Hemp's pass to the shot: 22.8 s
Time from throw-in to shot: 13.3 s
Sweden passes between throw-in and shot: 2 (completed: 2); including the throw-in: 3
England on-ball events between throw-in and shot (ball won back inside Sweden's possession):
   timestamp player_name          type outcome  under_pressure
00:24:28.392 Lauren Hemp Ball Recovery     NaN           False
00:24:28.392 Lauren Hemp         Carry     NaN            True
00:24:29.903 Lauren Hemp  Dispossessed     NaN            True


---
## H2 — "Substitutions after 70' improved England's chances"

**Data needed:** England shots with `shot_statsbomb_xg` and coordinates, flagged before/after 77';
cumulative xG per minute for both teams; summary before vs after 77'.

* Shots = `type == "Shot"` in periods 1–4 (the shootout is excluded; there were no in-play penalties).
* **Split point** = the exact timestamp of Kelly's substitution (77:33, period 2); `after_77 = True` for shots at or after it.
* **Inside the box** = shot location with x ≥ 102 and 18 ≤ y ≤ 62.
* The two windows have very different lengths (≈ 80 vs ≈ 53 played minutes), so the summary also gives per-90 rates.

In [17]:
shots = in_play[in_play.type == "Shot"].copy()
shots = add_split_flag(shots)
shots = d.split_xy(shots, "location", "shot")
shots["end_x"] = shots.shot_end_location.map(lambda v: v[0] if isinstance(v, list) else np.nan)
shots["end_y"] = shots.shot_end_location.map(lambda v: v[1] if isinstance(v, list) else np.nan)
shots["end_z"] = shots.shot_end_location.map(lambda v: v[2] if isinstance(v, list) and len(v) > 2 else np.nan)
shots["in_box"] = d.in_penalty_box(shots.shot_x, shots.shot_y)
shots["is_goal"] = shots.shot_outcome.eq("Goal")
shots["xg"] = shots.shot_statsbomb_xg

SHOT_COLS = [
    "id", "index", "period", "minute", "second", "display_minute", "team", "player", "player_name",
    "shot_type", "shot_outcome", "is_goal", "shot_body_part", "shot_technique", "play_pattern",
    "xg", "shot_x", "shot_y", "end_x", "end_y", "end_z", "in_box", "after_77",
]
h2_eng_shots = shots[shots.team == d.ENGLAND][SHOT_COLS].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h2_eng_shots, "h2_england_shots")))
h2_eng_shots

saved: data/processed/h2_england_shots.parquet


,id,index,period,minute,second,display_minute,team,player,player_name,shot_type,shot_outcome,is_goal,shot_body_part,shot_technique,play_pattern,xg,shot_x,shot_y,end_x,end_y,end_z,in_box,after_77
0,6602685e-dbc6-4d08-924b-a1bcb04eefc2,202,1,5,43,6,England Women's,Lauren Hemp,Lauren Hemp,Open Play,Saved to Post,False,Left Foot,Normal,From Free Kick,0.034543,103.2,24.5,119.0,39.4,2.1,True,False
1,efe41282-2470-4504-9bc1-c86b708b80e2,911,1,27,8,28,England Women's,Ella Toone,Ella Toone,Open Play,Blocked,False,Right Foot,Normal,From Counter,0.063342,101.4,43.0,103.1,42.9,NaN,False,False
2,ca87de0e-aac3-4f26-b196-552799167956,1280,1,38,30,39,England Women's,Lauren James,Lauren James,Open Play,Blocked,False,Right Foot,Normal,From Free Kick,0.028289,95.6,33.4,106.9,37.0,NaN,False,False
3,54030b01-e694-4f67-91d8-b4aab8281db2,1761,2,50,12,51,England Women's,Lauren Hemp,Lauren Hemp,Open Play,Off T,False,Head,Normal,Regular Play,0.079779,114.1,32.3,120.0,47.9,2.5,True,False
4,61b7c2bd-ad24-4c34-a872-83925638eba2,2065,2,58,18,59,England Women's,Lauren James,Lauren James,Open Play,Blocked,False,Left Foot,Normal,Regular Play,0.032129,97.1,42.4,98.5,42.2,NaN,False,False
5,3f6484a6-7200-4d79-9dfd-be4f80bc58d5,2079,2,58,59,59,England Women's,Georgia Stanway,Georgia Stanway,Open Play,Blocked,False,Right Foot,Half Volley,From Corner,0.040255,102.9,34.9,104.5,35.4,NaN,True,False
6,bc719170-2045-47ce-998b-d1d35b3d9fd2,2333,2,66,29,67,England Women's,Ella Toone,Ella Toone,Open Play,Blocked,False,Left Foot,Normal,From Throw In,0.057788,102.5,47.4,103.4,47.0,NaN,True,False
7,3d3e38fb-0892-48c0-9c22-809ec259fb77,2527,2,72,42,73,England Women's,Michelle Agyemang,Michelle Agyemang,Open Play,Blocked,False,Head,Normal,From Free Kick,0.032682,109.1,36.1,110.0,36.3,NaN,True,False
8,8226bca3-f6a7-498e-9f07-d88600c88777,2532,2,72,43,73,England Women's,Lauren Hemp,Lauren Hemp,Open Play,Saved,False,Head,Normal,From Free Kick,0.118530,114.4,36.7,118.8,37.4,2.2,True,False
9,3c2b187b-f16f-4e49-8998-71f42f9114d7,2628,2,75,30,76,England Women's,Michelle Agyemang,Michelle Agyemang,Open Play,Off T,False,Head,Normal,Regular Play,0.028826,109.9,31.5,120.0,32.7,6.7,True,False


In [18]:
# Cumulative xG per minute: one row per (period, minute) actually played, both teams.
grid = (
    in_play.groupby("period").minute.agg(["min", "max"])
    .apply(lambda r: list(range(r["min"], r["max"] + 1)), axis=1)
    .explode().rename("minute").reset_index()
)
grid["minute"] = grid.minute.astype(int)
grid["elapsed_index"] = range(len(grid))  # monotonic x-axis without stoppage overlap

per_min = (
    shots.pivot_table(index=["period", "minute"], columns="team", values="xg", aggfunc="sum")
    .reindex(columns=[d.SWEDEN, d.ENGLAND])
)
goals_min = (
    shots[shots.is_goal]
    .pivot_table(index=["period", "minute"], columns="team", values="is_goal", aggfunc="sum")
    .reindex(columns=[d.SWEDEN, d.ENGLAND])
)
h2_xg_timeline = grid.merge(per_min, left_on=["period", "minute"], right_index=True, how="left")
h2_xg_timeline = h2_xg_timeline.rename(columns={d.SWEDEN: "xg_sweden", d.ENGLAND: "xg_england"}).fillna(0.0)
h2_xg_timeline = h2_xg_timeline.merge(goals_min, left_on=["period", "minute"], right_index=True, how="left")
h2_xg_timeline = h2_xg_timeline.rename(columns={d.SWEDEN: "goals_sweden", d.ENGLAND: "goals_england"})
h2_xg_timeline[["goals_sweden", "goals_england"]] = h2_xg_timeline[["goals_sweden", "goals_england"]].fillna(0).astype(int)
for team in ["sweden", "england"]:
    h2_xg_timeline[f"cum_xg_{team}"] = h2_xg_timeline[f"xg_{team}"].cumsum()
    h2_xg_timeline[f"cum_goals_{team}"] = h2_xg_timeline[f"goals_{team}"].cumsum()
h2_xg_timeline["display_minute"] = h2_xg_timeline.minute + 1
h2_xg_timeline["after_77"] = (h2_xg_timeline.period * 1000 + h2_xg_timeline.minute + 1) > SPLIT_KEY
print("saved:", d.relative_path(d.save_processed(h2_xg_timeline, "h2_cumulative_xg_per_minute")))
h2_xg_timeline.tail()

saved: data/processed/h2_cumulative_xg_per_minute.parquet


,period,minute,elapsed_index,xg_sweden,xg_england,goals_sweden,goals_england,cum_xg_sweden,cum_goals_sweden,cum_xg_england,cum_goals_england,display_minute,after_77
129,4,119,129,0.0,0.000000,0,0,1.728191,2,1.776702,2,120,True
130,4,120,130,0.0,0.000000,0,0,1.728191,2,1.776702,2,121,True
131,4,121,131,0.0,0.038062,0,0,1.728191,2,1.814764,2,122,True
132,4,122,132,0.0,0.096353,0,0,1.728191,2,1.911117,2,123,True
133,4,123,133,0.0,0.000000,0,0,1.728191,2,1.911117,2,124,True


### H2 — numeric summary

Note: `after_77` in the per-minute table marks minutes that end after the split point
(the minute containing the substitution, 77:00–77:59, counts as *after*); the shot table uses exact timestamps.

In [19]:
def shot_summary(df: pd.DataFrame, minutes: float) -> pd.Series:
    return pd.Series({
        "minutes": minutes,
        "shots": len(df),
        "goals": int(df.is_goal.sum()),
        "xg": round(df.xg.sum(), 3),
        "xg_per_shot": round(df.xg.mean(), 3) if len(df) else np.nan,
        "share_in_box": round(df.in_box.mean(), 3) if len(df) else np.nan,
        "shots_per_90": round(len(df) / minutes * 90, 2),
        "xg_per_90": round(df.xg.sum() / minutes * 90, 3),
    })


rows = {}
for team, label in [(d.ENGLAND, "England"), (d.SWEDEN, "Sweden")]:
    t = shots[shots.team == team]
    rows[(label, "before 77'")] = shot_summary(t[~t.after_77], MIN_BEFORE)
    rows[(label, "after 77'")] = shot_summary(t[t.after_77], MIN_AFTER)
h2_summary = pd.DataFrame(rows).T
h2_summary.index.names = ["team", "window"]
d.save_processed(h2_summary.reset_index(), "h2_summary_before_after_77")
h2_summary

minutes  shots  goals     xg  xg_per_shot  share_in_box  shots_per_90  xg_per_90
team    window                                                                                      
England before 77'    79.17   10.0    0.0  0.516        0.052         0.700         11.37      0.587
        after 77'     52.68    8.0    2.0  1.395        0.174         0.875         13.67      2.383
Sweden  before 77'    79.17    6.0    2.0  1.215        0.203         1.000          6.82      1.381
        after 77'     52.68    7.0    0.0  0.513        0.073         0.857         11.96      0.877

### H2 — game-state windows

The before/after split above mixes very different phases. Below, the second half is cut at the substitution
timestamps taken from the data, so that the first three windows all **start** with Sweden leading 2-0:

| Window | From | To |
|---|---|---|
| `W1 control` | start of period 2 | England's first substitution (triple change; no England subs before) |
| `W2 triple sub` | England's first substitution (Mead, Agyemang, Morgan) | Kelly's substitution |
| `W3 Kelly on` | Kelly's substitution | end of period 2 |
| `W4 extra time` | start of period 3 | end of period 4 |

England's goals (78:31, 80:13) fall inside `W3`, so its scoreline changes during the window; `score_at_start`
and `goals` show this. `W2` lasts only about 8 minutes, so its per-90 rates rest on very few shots.

In [20]:
eng_subs = subs[subs.team == d.ENGLAND]
first_sub = eng_subs.iloc[0]
triple_subs = eng_subs[(eng_subs.period == first_sub.period)
                       & ((eng_subs.minute + eng_subs.second / 60) < SPLIT_CLOCK)]
print("England substitutions in W2 boundary group:",
      ", ".join(f"{r.player_on} ({int(r.minute)}:{int(r.second):02d})" for r in triple_subs.itertuples()))

TRIPLE_PERIOD, TRIPLE_CLOCK = int(first_sub.period), first_sub.minute + first_sub.second / 60
p_bounds = in_play.assign(c=d.clock(in_play)).groupby("period").c.agg(["min", "max"])

h2_windows_def = pd.DataFrame([
    ("W1 control", 2, p_bounds.loc[2, "min"], TRIPLE_PERIOD, TRIPLE_CLOCK),
    ("W2 triple sub", TRIPLE_PERIOD, TRIPLE_CLOCK, SPLIT_PERIOD, SPLIT_CLOCK),
    ("W3 Kelly on", SPLIT_PERIOD, SPLIT_CLOCK, 2, p_bounds.loc[2, "max"]),
    ("W4 extra time", 3, p_bounds.loc[3, "min"], 4, p_bounds.loc[4, "max"]),
], columns=["window", "start_period", "start_clock", "end_period", "end_clock"])
h2_windows_def["start_key"] = h2_windows_def.start_period * 1000 + h2_windows_def.start_clock
h2_windows_def["end_key"] = h2_windows_def.end_period * 1000 + h2_windows_def.end_clock
h2_windows_def["minutes"] = h2_windows_def.apply(
    lambda w: d.played_minutes(events, w.start_period, w.start_clock, w.end_period, w.end_clock), axis=1)

shot_key = d.time_key(shots)
goal_keys = shots[shots.is_goal].assign(k=shot_key)
rows = []
for w in h2_windows_def.itertuples():
    # Windows ending at the end of a period include that period's last event (inclusive end).
    ends_period = w.end_clock == p_bounds.loc[w.end_period, "max"]
    end_ok = (shot_key <= w.end_key) if ends_period else (shot_key < w.end_key)
    in_w = (shot_key >= w.start_key) & end_ok
    before = goal_keys[goal_keys.k < w.start_key].team.value_counts()
    score = f"{before.get(d.SWEDEN, 0)}-{before.get(d.ENGLAND, 0)}"
    for team, label in [(d.ENGLAND, "England"), (d.SWEDEN, "Sweden")]:
        t = shots[in_w & (shots.team == team)]
        rows.append({"window": w.window, "team": label, "score_at_start_swe_eng": score,
                     **shot_summary(t, w.minutes).to_dict()})
h2_windows = pd.DataFrame(rows)
h2_windows = h2_windows.merge(h2_windows_def[["window", "start_period", "start_clock", "end_period", "end_clock"]],
                              on="window")
print("saved:", d.relative_path(d.save_processed(h2_windows, "h2_summary_game_state_windows")))
h2_windows.drop(columns=["start_period", "start_clock", "end_period", "end_clock"]).set_index(["window", "team"])

England substitutions in W2 boundary group: Bethany Mead (69:09), Esme Beth Morgan (69:12), Michelle Agyemang (69:29)
saved: data/processed/h2_summary_game_state_windows.parquet


score_at_start_swe_eng  minutes  shots  goals     xg  xg_per_shot  share_in_box  shots_per_90  xg_per_90
window        team                                                                                                            
W1 control    England                    2-0    24.15    4.0    0.0  0.210        0.052         0.750         14.91      0.782
              Sweden                     2-0    24.15    2.0    0.0  0.449        0.224         1.000          7.45      1.672
W2 triple sub England                    2-0     8.40    3.0    0.0  0.180        0.060         1.000         32.14      1.929
              Sweden                     2-0     8.40    0.0    0.0  0.000          NaN           NaN          0.00      0.000
W3 Kelly on   England                    2-0    16.05    4.0    2.0  1.222        0.305         1.000         22.43      6.850
              Sweden                     2-0    16.05    1.0    0.0  0.086        0.086         1.000          5.61      0.480
W4 extra time England                    2-2    36.63    4.0    0.0  0.173        0.043         0.750          9.83      0.426
              Sweden                     2-2    36.63    6.0    0.0  0.427        0.071         0.833         14.74      1.050

In [21]:
# Sanity check: every second-half / extra-time shot falls into exactly one window.
n_windows = sum(h2_windows.shots)
n_shots = int((shots.period >= 2).sum())
print(f"Shots in windows: {n_windows:.0f}; shots in periods 2-4: {n_shots} -> "
      f"{'OK' if n_windows == n_shots else 'MISMATCH'}")

Shots in windows: 24; shots in periods 2-4: 24 -> OK


---
## H3 — "England switched to a 3-4-3"

**Data needed:** England completed passes in 0–70' and 77'–end of extra time, respecting stable-lineup
segments; average player positions and passer → receiver pairs with pass counts.

* **Completed pass** = `type == "Pass"` and `pass_outcome` is null (StatsBomb leaves it empty for completed passes).
* **Stable segments** come from `get_stable_lineup_windows(events, ENGLAND)` (a new segment at every substitution).
  * Window **"0-70"** = segments that end no later than 70:00 → in practice the starting XI until the first substitution at 69:09.
  * Window **"77-end"** = segments that start at or after Kelly's substitution (77:33): 77:33–103:42, 103:42–105:00 and 105:00–end of extra time.
  Each segment is kept separately (`segment` column) so a pass network is never drawn with more than 11 players.
  Segments shorter than `MIN_SEGMENT_MIN = 1` minute are dropped: the triple substitution at 69:09/69:12/69:29
  creates two segments of a few seconds each with a half-changed XI.
* **Set pieces** (`Throw-in`, `Corner`, `Free Kick`, `Goal Kick`, `Kick Off`) are flagged and, with
  `EXCLUDE_SET_PIECES = True`, left out of the average positions and pairs (they distort average positions).
* **Average position** = mean location of the passes a player *made* in the segment; `n_received` is also given.
* StatsBomb's own formation labels (`Starting XI` / `Tactical Shift` events) are saved too — note they do
  **not** say 3-4-3; the hypothesis has to be tested from positions.

> **Note on formations.** StatsBomb's only change of England formation in normal time is a `Tactical Shift`
> to **4-1-2-1-2 at 70:05**, right after the triple substitution; it reverts to 4-2-3-1 for extra time.
> StatsBomb records the nominal (out-of-possession / team-sheet) shape. The "3-4-3" claim refers to England's
> **in-possession** shape — e.g. one full-back staying back to form a back three while the other pushes into
> midfield — so it cannot be confirmed or rejected from the formation label. It has to be checked from the
> average positions and pass links in the windows below.

In [22]:
EXCLUDE_SET_PIECES = True
MIN_SEGMENT_MIN = 1.0
SET_PIECES = ["Throw-in", "Corner", "Free Kick", "Goal Kick", "Kick Off"]

windows = d.get_stable_lineup_windows(events, d.ENGLAND)
windows["window"] = np.select(
    [(windows.end_period * 1000 + windows.end_clock) <= 2000 + 70,
     (windows.start_period * 1000 + windows.start_clock) >= SPLIT_KEY],
    ["0-70", "77-end"], default="",
)
windows.loc[windows.duration_min < MIN_SEGMENT_MIN, "window"] = ""
windows.drop(columns="players")

,segment,start_period,start_clock,end_period,end_clock,duration_min,n_players,window
0,1,1,0.000000,2,69.150000,70.77,11,0-70
1,2,2,69.150000,2,69.200000,0.05,11,
2,3,2,69.200000,2,69.483333,0.28,11,
3,4,2,69.483333,2,77.550000,8.07,11,
4,5,2,77.550000,3,103.700000,29.75,11,77-end
5,6,3,103.700000,4,105.000000,4.82,11,77-end
6,7,4,105.000000,4,123.116667,18.12,11,77-end


In [23]:
passes = []
for _, seg in windows[windows.window != ""].iterrows():
    p = d.filter_segment(events, seg)
    p = p[(p.team == d.ENGLAND) & (p.type == "Pass") & p.pass_outcome.isna()].copy()
    p["segment"], p["window"] = seg.segment, seg.window
    passes.append(p)
h3_passes = pd.concat(passes)
h3_passes["set_piece"] = h3_passes.pass_type.isin(SET_PIECES)
h3_passes["recipient_name"] = h3_passes.pass_recipient.map(names).fillna(h3_passes.pass_recipient)
h3_passes = d.split_xy(h3_passes, "location", "pass")
h3_passes = d.split_xy(h3_passes, "pass_end_location", "pass_end")
h3_passes = h3_passes[[
    "id", "index", "segment", "window", "period", "minute", "second", "player", "player_name", "position",
    "pass_recipient", "recipient_name", "pass_type", "set_piece", "pass_height",
    "pass_x", "pass_y", "pass_end_x", "pass_end_y",
]].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h3_passes, "h3_england_completed_passes")))

net = h3_passes[~h3_passes.set_piece] if EXCLUDE_SET_PIECES else h3_passes

h3_positions = net.groupby(["window", "segment", "player"]).agg(
    player_name=("player_name", "first"),
    position=("position", lambda s: s.mode().iloc[0]),
    avg_x=("pass_x", "mean"), avg_y=("pass_y", "mean"), n_passes=("id", "size"),
).reset_index()
received = net.groupby(["window", "segment", "pass_recipient"]).size().rename("n_received")
h3_positions = h3_positions.merge(received, left_on=["window", "segment", "player"],
                                  right_index=True, how="left").fillna({"n_received": 0})
h3_positions["n_received"] = h3_positions.n_received.astype(int)
print("saved:", d.relative_path(d.save_processed(h3_positions, "h3_england_average_positions")))

h3_pairs = (
    net.groupby(["window", "segment", "player", "pass_recipient"]).size().rename("n_passes")
    .reset_index().rename(columns={"player": "passer", "pass_recipient": "recipient"})
    .sort_values(["segment", "n_passes"], ascending=[True, False])
)
h3_pairs["passer_name"] = h3_pairs.passer.map(names).fillna(h3_pairs.passer)
h3_pairs["recipient_name"] = h3_pairs.recipient.map(names).fillna(h3_pairs.recipient)
print("saved:", d.relative_path(d.save_processed(h3_pairs, "h3_england_pass_pairs")))

saved: data/processed/h3_england_completed_passes.parquet


saved: data/processed/h3_england_average_positions.parquet
saved: data/processed/h3_england_pass_pairs.parquet


In [24]:
# StatsBomb formation labels and assigned positions for England over time.
tac = events[(events.team == d.ENGLAND) & events.type.isin(["Starting XI", "Tactical Shift"])]
h3_formations = pd.DataFrame([
    dict(period=r.period, minute=r.minute, second=r.second, event=r.type,
         formation=str(r.tactics["formation"]), player=p["player"]["name"],
         player_name=names.get(p["player"]["name"], p["player"]["name"]),
         position=p["position"]["name"], jersey_number=p["jersey_number"])
    for r in tac.itertuples() for p in r.tactics["lineup"]
])
print("saved:", d.relative_path(d.save_processed(h3_formations, "h3_england_formations")))
h3_formations.groupby(["period", "minute", "second", "event"]).formation.first().reset_index()

saved: data/processed/h3_england_formations.parquet


,period,minute,second,event,formation
0,1,0,0,Starting XI,4231
1,1,28,35,Tactical Shift,4231
2,2,54,16,Tactical Shift,4231
3,2,70,5,Tactical Shift,41212
4,3,90,0,Tactical Shift,4231
5,4,105,0,Tactical Shift,4231


### H3 — numeric summary

Pass volume is reported per 90 so segments of different length can be compared. The **suggested minimum edge
threshold** (minimum passes for a passer → receiver link to be drawn) scales a full-match threshold of
`EDGE_THRESHOLD_PER_90 = 4` passes to the segment length, with a floor of 2 so that a single pass never becomes a
link: `max(2, round(4 * duration_min / 90))`. `pairs_at_threshold` shows how many links survive. The same is
given per window (all segments of the window pooled) for reference, but networks should be drawn per segment.

In [25]:
EDGE_THRESHOLD_PER_90 = 4
EDGE_THRESHOLD_FLOOR = 2


def edge_threshold(minutes: float) -> int:
    return max(EDGE_THRESHOLD_FLOOR, round(EDGE_THRESHOLD_PER_90 * minutes / 90))


seg_summary = windows[windows.window != ""][["segment", "window", "start_period", "start_clock",
                                             "end_period", "end_clock", "duration_min"]].copy()
seg_summary["completed_passes"] = seg_summary.segment.map(net.groupby("segment").size()).fillna(0).astype(int)
seg_summary["passes_per_min"] = (seg_summary.completed_passes / seg_summary.duration_min).round(2)
seg_summary["passes_per_90"] = (seg_summary.completed_passes / seg_summary.duration_min * 90).round(1)
seg_summary["edge_threshold"] = seg_summary.duration_min.map(edge_threshold)
seg_summary["pairs_at_threshold"] = [
    int((h3_pairs[h3_pairs.segment == r.segment].n_passes >= r.edge_threshold).sum())
    for r in seg_summary.itertuples()
]
print("saved:", d.relative_path(d.save_processed(seg_summary, "h3_segment_summary")))

win_summary = seg_summary.groupby("window").agg(
    segments=("segment", "size"), duration_min=("duration_min", "sum"),
    completed_passes=("completed_passes", "sum"),
)
win_summary["passes_per_90"] = (win_summary.completed_passes / win_summary.duration_min * 90).round(1)
win_summary["edge_threshold"] = win_summary.duration_min.map(edge_threshold)
print("saved:", d.relative_path(d.save_processed(win_summary.reset_index(), "h3_window_summary")))
print(win_summary.to_string(), "\n")

MIN_PASSES_FOR_NETWORK = 30
sparse = seg_summary[seg_summary.completed_passes < MIN_PASSES_FOR_NETWORK]
for r in sparse.itertuples():
    print(f"Warning: segment {r.segment} ({r.window}) has only {r.completed_passes} completed "
          f"passes in {r.duration_min} min - too few for a reliable pass network.")
seg_summary

saved: data/processed/h3_segment_summary.parquet
saved: data/processed/h3_window_summary.parquet
        segments  duration_min  completed_passes  passes_per_90  edge_threshold
window                                                                         
0-70           1         70.77               272          345.9               3
77-end         3         52.69                93          158.9               2 



,segment,window,start_period,start_clock,end_period,end_clock,duration_min,completed_passes,passes_per_min,passes_per_90,edge_threshold,pairs_at_threshold
0,1,0-70,1,0.00,2,69.150000,70.77,272,3.84,345.9,3,51
4,5,77-end,2,77.55,3,103.700000,29.75,76,2.55,229.9,2,17
5,6,77-end,3,103.70,4,105.000000,4.82,0,0.00,0.0,2,0
6,7,77-end,4,105.00,4,123.116667,18.12,17,0.94,84.4,2,1


In [26]:
# StatsBomb-assigned positions at the start and after every Tactical Shift.
pivot = h3_formations.assign(
    t=lambda x: "P" + x.period.astype(str) + " " + x.minute.astype(str).str.zfill(3)
    + ":" + x.second.astype(str).str.zfill(2) + " " + x.formation
)
pivot.pivot_table(index="player_name", columns="t", values="position", aggfunc="first").fillna("-")

t,P1 000:00 4231,P1 028:35 4231,P2 054:16 4231,P2 070:05 41212,P3 090:00 4231,P4 105:00 4231
player_name,,,,,,
Alessia Russo,Center Forward,Center Forward,Center Forward,Right Center Forward,Center Attacking Midfield,Center Attacking Midfield
Alex Greenwood,Left Back,Left Back,Left Back,Left Back,Left Back,Left Center Back
Bethany Mead,-,-,-,Right Center Midfield,Right Wing,Right Defensive Midfield
Chloe Kelly,-,-,-,-,Left Wing,Left Wing
Ella Toone,Center Attacking Midfield,Center Attacking Midfield,Center Attacking Midfield,-,-,-
Esme Morgan,-,-,-,Left Center Back,Left Center Back,Right Center Back
Georgia Stanway,Left Defensive Midfield,Left Defensive Midfield,Left Defensive Midfield,-,-,-
Grace Clinton,-,-,-,-,-,Left Defensive Midfield
Hannah Hampton,Goalkeeper,Goalkeeper,Goalkeeper,Goalkeeper,Goalkeeper,Goalkeeper


In [27]:
for seg in seg_summary.segment:
    top = h3_pairs[h3_pairs.segment == seg].head(5)
    print(f"Segment {seg} top pairs:")
    print(top[["passer_name", "recipient_name", "n_passes"]].to_string(index=False), "\n")

Segment 1 top pairs:
    passer_name  recipient_name  n_passes
 Alex Greenwood Leah Williamson        11
 Hannah Hampton Leah Williamson        10
 Jessica Carter  Hannah Hampton        10
Leah Williamson  Jessica Carter         8
Leah Williamson  Alex Greenwood         7 

Segment 5 top pairs:
   passer_name recipient_name  n_passes
   Esme Morgan Alex Greenwood         8
   Chloe Kelly   Lauren James         4
Alex Greenwood    Esme Morgan         3
   Keira Walsh    Chloe Kelly         3
  Lauren James    Chloe Kelly         3 

Segment 6 top pairs:
Empty DataFrame
Columns: [passer_name, recipient_name, n_passes]
Index: [] 

Segment 7 top pairs:
   passer_name recipient_name  n_passes
   Lucy Bronze Hannah Hampton         2
 Alessia Russo Alex Greenwood         1
 Alessia Russo    Lucy Bronze         1
Alex Greenwood   Bethany Mead         1
Alex Greenwood   Lauren James         1 



---
## H4 — "England's crossing strategy worked"

**Data needed:** England crosses before/after 77' with flank, completion and passer.

* **Cross** = `type == "Pass"` and `pass_cross == True` (StatsBomb's own cross tag), periods 1–4.
  Corners tagged as crosses are kept and flagged with `pass_type`.
* **Completed** = `pass_outcome` is null.
* **Flank** from the cross origin in England's attacking frame: `left` if y < 40, else `right`.
* **Led to shot** = `pass_shot_assist` or `pass_goal_assist` is true.

In [28]:
crosses = in_play[(in_play.team == d.ENGLAND) & (in_play.type == "Pass") & in_play.pass_cross.eq(True)].copy()
crosses = add_split_flag(crosses)
crosses = d.split_xy(crosses, "location", "cross")
crosses = d.split_xy(crosses, "pass_end_location", "cross_end")
crosses["flank"] = np.where(crosses.cross_y < 40, "left", "right")
crosses["completed"] = crosses.pass_outcome.isna()
crosses["led_to_shot"] = crosses.pass_shot_assist.eq(True) | crosses.pass_goal_assist.eq(True)
crosses["goal_assist"] = crosses.pass_goal_assist.eq(True)
crosses["recipient_name"] = crosses.pass_recipient.map(names).fillna(crosses.pass_recipient)
crosses["swing"] = np.select(
    [crosses.pass_inswinging.eq(True), crosses.pass_outswinging.eq(True)], ["inswinging", "outswinging"], ""
)
h4_crosses = crosses[[
    "id", "index", "period", "minute", "second", "display_minute", "player", "player_name", "position",
    "pass_recipient", "recipient_name", "pass_type", "pass_height", "pass_body_part", "swing",
    "pass_outcome", "completed", "led_to_shot", "goal_assist",
    "cross_x", "cross_y", "cross_end_x", "cross_end_y", "flank", "after_77",
]].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h4_crosses, "h4_england_crosses")))
h4_crosses.head()

saved: data/processed/h4_england_crosses.parquet


,id,index,period,minute,second,display_minute,player,player_name,position,pass_recipient,recipient_name,pass_type,pass_height,pass_body_part,swing,pass_outcome,completed,led_to_shot,goal_assist,cross_x,cross_y,cross_end_x,cross_end_y,flank,after_77
0,425f5014-88d4-4859-82ca-2bcdb3887cb5,196,1,5,40,6,Georgia Stanway,Georgia Stanway,Left Defensive Midfield,Keira Walsh,Keira Walsh,NaN,Ground Pass,Right Foot,,NaN,True,False,False,115.0,7.3,108.1,30.7,left,False
1,339500fa-1239-4005-acdd-c82f998b14d7,210,1,5,51,6,Lauren James,Lauren James,Right Wing,NaN,NaN,NaN,High Pass,Left Foot,,Incomplete,False,False,False,106.6,60.5,112.8,32.9,right,False
2,ffd93e6d-d20b-4b11-ab21-fab936037b80,493,1,13,36,14,Lucy Bronze,Lucy Bronze,Right Back,Lauren Hemp,Lauren Hemp,NaN,High Pass,Right Foot,,Unknown,False,False,False,88.8,75.4,107.7,35.5,right,False
3,d097b562-1a23-46d5-bfef-2d91fa46bc3e,717,1,20,8,21,Lauren Hemp,Lauren Hemp,Left Wing,NaN,NaN,NaN,High Pass,Left Foot,,Out,False,False,False,111.7,18.6,119.9,50.6,left,False
4,ceeed6d8-3c2f-495d-b333-db11954ae45d,1108,1,32,38,33,Lauren James,Lauren James,Right Wing,Lauren Hemp,Lauren Hemp,NaN,High Pass,Right Foot,,NaN,True,False,False,116.6,77.0,105.1,34.7,right,False


### H4 — numeric summary

In [29]:
def cross_summary(df: pd.DataFrame, minutes: float) -> pd.Series:
    return pd.Series({
        "minutes": minutes,
        "crosses": len(df),
        "completed": int(df.completed.sum()),
        "completion_rate": round(df.completed.mean(), 3) if len(df) else np.nan,
        "led_to_shot": int(df.led_to_shot.sum()),
        "goal_assists": int(df.goal_assist.sum()),
        "crosses_per_90": round(len(df) / minutes * 90, 2),
        "left": int((df.flank == "left").sum()),
        "right": int((df.flank == "right").sum()),
    })


h4_summary = pd.DataFrame({
    "before 77'": cross_summary(h4_crosses[~h4_crosses.after_77], MIN_BEFORE),
    "after 77'": cross_summary(h4_crosses[h4_crosses.after_77], MIN_AFTER),
}).T
d.save_processed(h4_summary.rename_axis("window").reset_index(), "h4_summary_before_after_77")
h4_summary

,minutes,crosses,completed,completion_rate,led_to_shot,goal_assists,crosses_per_90,left,right
before 77',79.17,22.0,8.0,0.364,3.0,0.0,25.01,13.0,9.0
after 77',52.68,12.0,6.0,0.500,3.0,1.0,20.50,7.0,5.0


In [30]:
h4_crosses.groupby(["after_77", "player_name"]).agg(
    crosses=("id", "size"), completed=("completed", "sum"), led_to_shot=("led_to_shot", "sum")
).sort_values(["after_77", "crosses"], ascending=[True, False])

crosses  completed  led_to_shot
after_77 player_name                                     
False    Lauren Hemp            6          1            1
         Alex Greenwood         4          1            1
         Lauren James           4          3            1
         Lucy Bronze            3          1            0
         Georgia Stanway        2          1            0
         Alessia Russo          1          0            0
         Ella Toone             1          1            0
         Keira Walsh            1          0            0
True     Chloe Kelly            5          4            3
         Alex Greenwood         4          1            0
         Lucy Bronze            2          1            0
         Lauren James           1          0            0

In [31]:
# Goal assists from crosses (England's two goals).
h4_crosses[h4_crosses.goal_assist][["display_minute", "player_name", "recipient_name", "flank", "pass_height"]]

,display_minute,player_name,recipient_name,flank,pass_height
23,79,Chloe Kelly,Lucy Bronze,left,High Pass


---
## H5 — "Hampton kept England alive"

**Data needed:** Sweden shots on target with xG and outcome; Sweden's total xG vs goals scored.

* Periods 1–4 only (the shootout is H6).
* **On target** = `shot_outcome` in `Goal`, `Saved`, `Saved to Post`.
  Not on target: `Off T`, `Post` (hit the woodwork without a save), `Blocked` (stopped by an outfield player)
  and `Saved Off Target` (the keeper touched a shot that was going wide — a save, but not a shot on target).
* Hampton's response is taken from the England `Goal Keeper` event linked to each shot through `related_events`.
* StatsBomb Open Data has no post-shot xG, so "xG prevented" below uses pre-shot xG of saved shots — a rough proxy.

In [32]:
ON_TARGET = ["Goal", "Saved", "Saved to Post"]

gk = events[(events.type == "Goal Keeper") & (events.team == d.ENGLAND)]
gk_by_shot = {}
for r in gk.itertuples():
    for rel in (r.related_events or []):
        gk_by_shot[rel] = (r.player, r.goalkeeper_type, r.goalkeeper_outcome)

swe_shots = shots[shots.team == d.SWEDEN].copy()
swe_shots["on_target"] = swe_shots.shot_outcome.isin(ON_TARGET)
gk_info = swe_shots.id.map(gk_by_shot)
swe_shots["goalkeeper"] = gk_info.map(lambda v: v[0] if isinstance(v, tuple) else None)
swe_shots["goalkeeper_type"] = gk_info.map(lambda v: v[1] if isinstance(v, tuple) else None)
swe_shots["goalkeeper_outcome"] = gk_info.map(lambda v: v[2] if isinstance(v, tuple) else None)

H5_COLS = SHOT_COLS + ["on_target", "goalkeeper", "goalkeeper_type", "goalkeeper_outcome"]
h5_swe_shots = swe_shots[H5_COLS].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h5_swe_shots, "h5_sweden_shots")))
h5_on_target = h5_swe_shots[h5_swe_shots.on_target].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h5_on_target, "h5_sweden_shots_on_target")))
h5_on_target[["display_minute", "player_name", "xg", "shot_outcome", "goalkeeper_type", "goalkeeper_outcome", "in_box"]]

saved: data/processed/h5_sweden_shots.parquet
saved: data/processed/h5_sweden_shots_on_target.parquet


,display_minute,player_name,xg,shot_outcome,goalkeeper_type,goalkeeper_outcome,in_box
0,2,Kosovare Asllani,0.262062,Goal,Goal Conceded,No Touch,True
1,25,Stina Blackstenius,0.286011,Goal,Goal Conceded,No Touch,True
2,46,Fridolina Rolfö,0.082706,Saved,Shot Saved,Touched Out,True
3,55,Stina Blackstenius,0.391173,Saved,Shot Saved,Saved Twice,True
4,88,Madelen Fatimma Maria Janogy,0.085677,Saved,Shot Saved,Touched Out,True


### H5 — numeric summary

In [33]:
saved = h5_swe_shots[h5_swe_shots.shot_outcome.isin(["Saved", "Saved to Post", "Saved Off Target"])]
h5_summary = pd.Series({
    "shots": len(h5_swe_shots),
    "shots_on_target": int(h5_swe_shots.on_target.sum()),
    "goals": int(h5_swe_shots.is_goal.sum()),
    "total_xg": round(h5_swe_shots.xg.sum(), 3),
    "goals_minus_xg": round(h5_swe_shots.is_goal.sum() - h5_swe_shots.xg.sum(), 3),
    "xg_on_target": round(h5_on_target.xg.sum(), 3),
    "saves_by_hampton": len(saved),
    "xg_of_saved_shots": round(saved.xg.sum(), 3),
    "blocked_shots": int(h5_swe_shots.shot_outcome.eq("Blocked").sum()),
    "xg_blocked": round(h5_swe_shots.loc[h5_swe_shots.shot_outcome.eq("Blocked"), "xg"].sum(), 3),
})
d.save_processed(h5_summary.rename("value").rename_axis("metric").reset_index(), "h5_summary")
h5_summary

shots                13.000
shots_on_target       5.000
goals                 2.000
total_xg              1.728
goals_minus_xg        0.272
xg_on_target          1.108
saves_by_hampton      4.000
xg_of_saved_shots     0.574
blocked_shots         5.000
xg_blocked            0.505
dtype: float64

In [34]:
h5_swe_shots.groupby("period").agg(shots=("id", "size"), on_target=("on_target", "sum"),
                                   xg=("xg", "sum"), goals=("is_goal", "sum")).round(3)

,shots,on_target,xg,goals
period,,,,
1,4,3,0.766,2
2,3,2,0.534,0
3,3,0,0.152,0
4,3,0,0.276,0


---
## H6 — "The shootout was mostly luck"

**Data needed:** shootout kicks (period 5) in order — taker, team, outcome, end location; a baseline
in-game penalty conversion across all Euro 2025 matches in the open data.

* Kicks = `type == "Shot"` in period 5, ordered by event index. Scored = `shot_outcome == "Goal"`.
* The goalkeeper is taken from the opposing `Goal Keeper` event linked through `related_events`.
* **Baseline A — in-game penalties, Euro 2025:** every `shot_type == "Penalty"` in periods 1–4 of all matches in
  competition 53 / season 315 (events cached per match in `data/raw/`).
* **Baseline B — women's shootouts:** every period-5 kick in all women's competitions in StatsBomb Open Data
  (`competition_gender == "female"` in `competitions.json`), **excluding this match**. A shootout needs a level
  score after the match, and league `Regular Season` games never go to penalties, so events are loaded only for
  level-score matches outside the regular season; the presence of period 5 is then checked in the event data.
* StatsBomb assigns every penalty the same xG (≈ 0.78), so xG adds nothing beyond the conversion rate here.
* The binomial probabilities assume independent kicks with the baseline conversion rate — a deliberately simple
  "how unusual was this?" benchmark, not a model of the shootout.

In [35]:
so = events[(events.period == d.SHOOTOUT_PERIOD) & (events.type == "Shot")].sort_values("index").copy()
so["kick_no"] = range(1, len(so) + 1)
so["scored"] = so.shot_outcome.eq("Goal")
so["end_x"] = so.shot_end_location.map(lambda v: v[0])
so["end_y"] = so.shot_end_location.map(lambda v: v[1])
so["end_z"] = so.shot_end_location.map(lambda v: v[2] if len(v) > 2 else np.nan)
so["round"] = so.groupby("team").cumcount() + 1

gk_all = events[events.type == "Goal Keeper"]
gk_map = {rel: r.player for r in gk_all.itertuples() for rel in (r.related_events or [])}
so["goalkeeper"] = so.id.map(gk_map)
so["england_score"] = (so.scored & so.team.eq(d.ENGLAND)).cumsum()
so["sweden_score"] = (so.scored & so.team.eq(d.SWEDEN)).cumsum()

h6_shootout = so[[
    "kick_no", "round", "team", "player", "player_name", "shot_outcome", "scored", "shot_body_part",
    "goalkeeper", "end_x", "end_y", "end_z", "shot_statsbomb_xg", "england_score", "sweden_score",
]].reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h6_shootout, "h6_shootout_kicks")))
h6_shootout

saved: data/processed/h6_shootout_kicks.parquet


,kick_no,round,team,player,player_name,shot_outcome,scored,shot_body_part,goalkeeper,end_x,end_y,end_z,shot_statsbomb_xg,england_score,sweden_score
0,1,1,England Women's,Alessia Russo,Alessia Russo,Goal,True,Right Foot,Jennifer Miley Falk,120.0,36.8,1.3,0.7835,1,0
1,2,1,Sweden Women's,Ingrid Filippa Angeldal,Ingrid Filippa Angeldal,Saved,False,Right Foot,Hannah Hampton,119.4,41.9,0.9,0.7835,1,0
2,3,2,England Women's,Lauren James,Lauren James,Saved,False,Right Foot,Jennifer Miley Falk,118.9,37.4,1.0,0.7835,1,0
3,4,2,Sweden Women's,Julia Zigiotti-Olme,Julia Zigiotti-Olme,Goal,True,Right Foot,Hannah Hampton,120.0,37.2,2.0,0.7835,1,1
4,5,3,England Women's,Bethany Mead,Bethany Mead,Saved,False,Right Foot,Jennifer Miley Falk,118.9,37.6,1.1,0.7835,1,1
5,6,3,Sweden Women's,Magdalena Lilly Eriksson,Magdalena Eriksson,Post,False,Left Foot,Hannah Hampton,120.0,44.2,0.4,0.7835,1,1
6,7,4,England Women's,Alex Greenwood,Alex Greenwood,Saved,False,Left Foot,Jennifer Miley Falk,119.0,41.7,1.0,0.7835,1,1
7,8,4,Sweden Women's,Gun Nathalie Björn,Gun Nathalie Björn,Goal,True,Right Foot,Hannah Hampton,120.0,42.9,1.2,0.7835,1,2
8,9,5,England Women's,Chloe Kelly,Chloe Kelly,Goal,True,Right Foot,Jennifer Miley Falk,120.0,43.1,1.6,0.7835,2,2
9,10,5,Sweden Women's,Jennifer Miley Falk,Jennifer Miley Falk,Off T,False,Right Foot,Hannah Hampton,120.0,38.4,4.0,0.7835,2,2


In [36]:
all_events = d.load_competition_events()  # cached per match after the first run
matches = d.load_matches()
pens = all_events[(all_events.type == "Shot") & (all_events.shot_type == "Penalty")].copy()
pens = pens.merge(matches[["match_id", "match_date", "home_team", "away_team", "competition_stage"]], on="match_id")
pens["in_game"] = pens.period < d.SHOOTOUT_PERIOD
pens["scored"] = pens.shot_outcome.eq("Goal")
pens["this_match"] = pens.match_id.eq(match_id)
h6_penalties = pens[[
    "match_id", "match_date", "competition_stage", "home_team", "away_team", "period", "minute",
    "team", "player", "shot_outcome", "scored", "in_game", "this_match", "shot_statsbomb_xg",
]].sort_values(["match_date", "match_id", "period", "minute"]).reset_index(drop=True)
print(f"matches loaded: {all_events.match_id.nunique()}")
print("saved:", d.relative_path(d.save_processed(h6_penalties, "h6_euro2025_penalties")))

matches loaded: 31
saved: data/processed/h6_euro2025_penalties.parquet


In [37]:
# Baseline B: shootout kicks in all women's competitions in the open data.
comps = d.load_competitions()
women = comps[comps.competition_gender == "female"]
print("Women's competition seasons in StatsBomb Open Data:")
print(women[["competition_id", "season_id", "competition_name", "season_name"]].to_string(index=False), "\n")

w_matches = pd.concat(
    [d.load_matches(r.competition_id, r.season_id) for r in women.itertuples()], ignore_index=True
)
candidates = w_matches[(w_matches.home_score == w_matches.away_score)
                       & (w_matches.competition_stage != "Regular Season")]
print(f"women's matches: {len(w_matches)}; level-score non-regular-season candidates: {len(candidates)}")

so_kicks = []
for r in candidates.itertuples():
    ev_m = d.load_events(r.match_id)
    k = ev_m[(ev_m.period == d.SHOOTOUT_PERIOD) & (ev_m.type == "Shot")]
    if len(k):
        so_kicks.append(k.assign(competition=r.competition, season=r.season, competition_stage=r.competition_stage,
                                 home_team=r.home_team, away_team=r.away_team, match_date=r.match_date))
h6_women_so = pd.concat(so_kicks, ignore_index=True)
h6_women_so["scored"] = h6_women_so.shot_outcome.eq("Goal")
h6_women_so["this_match"] = h6_women_so.match_id.eq(match_id)
h6_women_so = h6_women_so[[
    "match_id", "competition", "season", "competition_stage", "match_date", "home_team", "away_team",
    "index", "team", "player", "shot_outcome", "scored", "this_match",
]].sort_values(["match_date", "match_id", "index"]).reset_index(drop=True)
print("saved:", d.relative_path(d.save_processed(h6_women_so, "h6_women_shootout_kicks")))
h6_women_so.groupby(["competition", "season", "competition_stage", "home_team", "away_team", "this_match"]).agg(
    kicks=("scored", "size"), scored=("scored", "sum")
)

Women's competition seasons in StatsBomb Open Data:
 competition_id  season_id        competition_name season_name
             37        281 FA Women's Super League   2023/2024
             37         90 FA Women's Super League   2020/2021
             37         42 FA Women's Super League   2019/2020
             37          4 FA Women's Super League   2018/2019
            135        281       Frauen Bundesliga   2023/2024
            182        281                  Liga F   2023/2024
             49        107                    NWSL        2023
             49          3                    NWSL        2018
            131        281           Serie A Women   2023/2024
             53        315       UEFA Women's Euro        2025
             53        106       UEFA Women's Euro        2022
             72        107       Women's World Cup        2023
             72         30       Women's World Cup        2019 

women's matches: 1310; level-score non-regular-season candidates

saved: data/processed/h6_women_shootout_kicks.parquet


kicks  scored
competition                       season competition_stage home_team         away_team             this_match               
Europe - UEFA Women's Euro        2025   Final             England Women's   Spain Women's         False           9       4
                                         Quarter-finals    France Women's    Germany Women's       False          14      11
                                                           Sweden Women's    England Women's       True           14       5
International - Women's World Cup 2019   Round of 16       Norway Women's    Australia Women's     False           7       5
                                  2023   Quarter-finals    Australia Women's France Women's        False          20      13
                                         Round of 16       England Women's   Nigeria Women's       False           9       6
                                                           Sweden Women's    United States Women's False          14       9

### H6 — numeric summary

In [38]:
def conv(df: pd.DataFrame) -> str:
    return f"{int(df.scored.sum())}/{len(df)} = {df.scored.mean():.1%}" if len(df) else "n/a"


in_game = h6_penalties[h6_penalties.in_game]
other_so = h6_penalties[~h6_penalties.in_game & ~h6_penalties.this_match]
this_so = h6_penalties[~h6_penalties.in_game & h6_penalties.this_match]
print(f"In-game penalties, Euro 2025 (baseline): {conv(in_game)}")
print(f"Shootout kicks, other Euro 2025 matches: {conv(other_so)} "
      f"({other_so.match_id.nunique()} shootout(s))")
print(f"Shootout kicks, Sweden vs England:       {conv(this_so)}")
for team in [d.ENGLAND, d.SWEDEN]:
    print(f"  {team}: {conv(this_so[this_so.team == team])}")

def binom_cdf(k: int, n: int, p: float) -> float:
    return sum(comb(n, i) * p**i * (1 - p) ** (n - i) for i in range(k + 1))


n, k = len(this_so), int(this_so.scored.sum())
baseline_b = h6_women_so[~h6_women_so.this_match]
print(f"Women's shootouts, all open data (excl. this match): {conv(baseline_b)} "
      f"({baseline_b.match_id.nunique()} shootouts)\n")

h6_summary = pd.DataFrame([
    {"baseline": name, "kicks": len(df), "goals": int(df.scored.sum()),
     "matches": df.match_id.nunique(), "conversion": round(df.scored.mean(), 3),
     "expected_goals_in_14": round(n * df.scored.mean(), 2), "actual_goals": k,
     f"p_at_most_{k}_of_{n}": round(binom_cdf(k, n, df.scored.mean()), 4)}
    for name, df in [("A: in-game penalties, Euro 2025", in_game),
                     ("B: women's shootout kicks, open data (excl. this match)", baseline_b)]
])
print("saved:", d.relative_path(d.save_processed(h6_summary, "h6_summary")))

for team in [d.ENGLAND, d.SWEDEN]:
    t = this_so[this_so.team == team]
    print(f"{team}: {int(t.scored.sum())}/{len(t)}; "
          + "; ".join(f"P(<= {int(t.scored.sum())} of {len(t)} | {r.baseline[:1]}) = "
                      f"{binom_cdf(int(t.scored.sum()), len(t), r.conversion):.3f}"
                      for r in h6_summary.itertuples()))
h6_summary

In-game penalties, Euro 2025 (baseline): 8/14 = 57.1%
Shootout kicks, other Euro 2025 matches: 15/23 = 65.2% (2 shootout(s))
Shootout kicks, Sweden vs England:       5/14 = 35.7%
  England Women's: 3/7 = 42.9%
  Sweden Women's: 2/7 = 28.6%
Women's shootouts, all open data (excl. this match): 48/73 = 65.8% (6 shootouts)

saved: data/processed/h6_summary.parquet
England Women's: 3/7; P(<= 3 of 7 | A) = 0.348; P(<= 3 of 7 | B) = 0.187
Sweden Women's: 2/7; P(<= 2 of 7 | A) = 0.127; P(<= 2 of 7 | B) = 0.050


,baseline,kicks,goals,matches,conversion,expected_goals_in_14,actual_goals,p_at_most_5_of_14
0,"A: in-game penalties, Euro 2025",14,8,12,0.571,8.00,5,0.0892
1,"B: women's shootout kicks, open data (excl. th...",73,48,6,0.658,9.21,5,0.0210


In [39]:
h6_penalties.groupby(["in_game", "match_id", "home_team", "away_team"]).agg(
    kicks=("scored", "size"), scored=("scored", "sum")
)

kicks  scored
in_game match_id home_team           away_team                         
False   4018355  Sweden Women's      England Women's         14       5
        4018357  France Women's      Germany Women's         14      11
        4020846  England Women's     Spain Women's            9       4
True    3998837  Switzerland Women's Norway Women's           1       0
        3998849  Germany Women's     Denmark Women's          1       1
        3998850  France Women's      Wales W                  1       1
        3998852  WNT Finland         Switzerland Women's      1       1
        3998853  Norway Women's      Iceland Women's          1       1
        3998856  Sweden Women's      Germany Women's          1       1
        3998858  England Women's     Wales W                  1       1
        3998859  Netherlands Women's France Women's           1       1
        4018354  Norway Women's      Italy Women's            1       0
        4018356  Spain Women's       Switzerland Women's      2       0
        4018357  France Women's      Germany Women's          2       1
        4020005  England Women's     Italy Women's            1       0

---
## Saved tables

All tables are in `data/processed/` (parquet, not tracked by git). Load them with
`src.data.load_processed("<name>")`; nested StatsBomb columns are restored as Python lists/dicts.

In [40]:
for path in sorted(d.PROCESSED_DIR.glob("*.parquet")):
    print(f"{path.name:50} {len(d.load_processed(path.stem)):>6} rows")

h1_england_losses_first_half.parquet                   81 rows
h1_england_turnovers_first_half.parquet                36 rows
h1_goal_sequence_02min.parquet                         17 rows
h1_goal_sequence_25min.parquet                         22 rows
h1_sweden_defensive_actions_first_half.parquet         90 rows
h2_cumulative_xg_per_minute.parquet                   134 rows
h2_england_shots.parquet                               18 rows
h2_summary_before_after_77.parquet                      4 rows
h2_summary_game_state_windows.parquet                   8 rows
h3_england_average_positions.parquet                   31 rows
h3_england_completed_passes.parquet                   419 rows
h3_england_formations.parquet                          66 rows
h3_england_pass_pairs.parquet                         148 rows
h3_segment_summary.parquet                              4 rows
h3_window_summary.parquet                               2 rows
h4_england_crosses.parquet                             